Middleware:

Tracking Agent behaviour with logging, analytics and debugging 

Transforming prompts, tool selection, and output formatting 

Adding retries, fallbacks, and early termination logic

Applying rate limits, guardrails, and PII detectinon 

In [28]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

load_dotenv(override=True)

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001DDAD1C0640>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DDAD1C0FC0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Summarization Middleware

Automatically summarizes conversation history when approching token limits, preventing recent messages while compressing older context. Summarization is useful for the following:

- Long running converstaion that exceed context windows
- Multi- turn dialogues with extensive history 
- Applications where preserving full converstion context matters 

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# InMemorySaver is a checkpointer: it stores an agent’s conversation state in your computer’s 
# RAM so the agent can remember earlier messages.
# Message based summarization

# Add Middleware in the agent, set when the middleware need to trigger and the limits
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(), 
    middleware=[
        SummarizationMiddleware(
            model=model, # Use cheap models for summarization Purposes 
            trigger=("messages", 10), # Means messages length becomes 10
            keep=("messages", 4) # Keep the latest 4 messages 
        )
    ]
)

In [30]:
### Run with thread_id
config = {"configurable":{"thread_id":"test-1"}} # Label for the conversation 

In [31]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

# Here the 2 which we are getting is 1 for the human Message and 1 for the AI Message 
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config=config)
    print(f"Messages:{response}")
    print(f"Messages: {len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='e89c6f46-c6c7-4a19-9c56-f751a4e494a0'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 78, 'total_tokens': 111, 'completion_time': 0.067969619, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.003075617, 'prompt_tokens_details': None, 'queue_time': 0.397860008, 'total_time': 0.071045236}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17478-549b-70be-88ba-a11f22926dc0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 33, 'total_tokens': 111, 'output_token_details': {'reasoning': 14}})]}
Messages: 2
Me

### Token Size

In [32]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def Search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotles in {city}:
    1. Grand Hotel - 5 Star, $350/night, spa, pool, gym
    2. City Inn - 4 str, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model=model, 
    tools=[Search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 4000), # ("fraction", 0.005) # every 0.5% = ~640 tokens (Based on the context of LLM model it is variable )
            keep=("tokens", 200), # ("fraction", 0.002) # every 0.2% = ~256 tokens
            # Fraction = tokens/ 128000 # Different as per models 
        ),
    ]
)

In [33]:
config = {"configurable":{"thread_id":"test-2"}}

In [34]:
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 char as 1 token  

In [35]:
cities = ["Paris", "tokyo", "London", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages") 
    print(f"{(response['messages'])}")
    

Paris: ~437 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='52080737-8d08-4b48-b23e-f7551ee71799'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. We have a function Search_hotels(city). Likely we should call it.', 'tool_calls': [{'id': 'fc_50e477c8-e7f4-406f-9647-05c6269e09ad', 'function': {'arguments': '{"city":"Paris"}', 'name': 'Search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 129, 'total_tokens': 183, 'completion_time': 0.112180198, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005381345, 'prompt_tokens_details': None, 'queue_time': 0.395310959, 'total_time': 0.117561543}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1747

### Human in Loop Middleware

In [65]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_mail_tool(email_id: str) -> str:
    """Read recent emails from the given mail folder and return a summary of them."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email to the given recipient with a subject and body."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [84]:
agent = create_agent(
    model = model, 
    tools = [read_mail_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    system_prompt="If a reviewer edits a tool call, tell the user exactly what was executed instead.",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_mail_tool": False,
            }
        )
    ]
)

In [85]:
config = {"configurable": {"thread_id":"test-approve"}}
result = agent.invoke(
    {"messages" : [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [86]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a0794676-8c6c-42dc-8cad-a10031467d70'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with given parameters.', 'tool_calls': [{'id': 'fc_5772f2e0-2854-4cee-97c8-098654767120', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 205, 'total_tokens': 263, 'completion_time': 0.121790767, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.008688857, 'prompt_tokens_details': None, 'queue_time': 0.213724979, 'total_time': 0.130479624}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5a93aea882', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

In [87]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused Approving")

    result = agent.invoke(
        Command( # This command library executes a command 
            resume={
                "decisions":[
                    {"type":"approve"} # If we write reject, it will get rejected 
                ]
            }
        ),
        config=config
    )

    print(f"Result : {result['messages'][-1].content}")

Paused Approving
Result : Your email has been sent to **john@test.com** with the subject **“Hello.”**


In [88]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a0794676-8c6c-42dc-8cad-a10031467d70'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with given parameters.', 'tool_calls': [{'id': 'fc_5772f2e0-2854-4cee-97c8-098654767120', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 205, 'total_tokens': 263, 'completion_time': 0.121790767, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.008688857, 'prompt_tokens_details': None, 'queue_time': 0.213724979, 'total_time': 0.130479624}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5a93aea882', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

In [89]:
config = {"configurable": {"thread_id":"test-edit2"}}
result = agent.invoke(
    {"messages" : [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [90]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused Approving")

    result = agent.invoke(
        Command( # This command library executes a command 
            resume={
                "decisions":[
                    {"type":"edit",
                     "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "bob@example.com",
                            "subject": "Running late",
                            "body": "I'll be 10 minutes late.",
                        }
                    }
                    } # If we edit
                ]
            }
        ),
        config=config
    )

    print(f"Result : {result['messages'][-1].content}")

Paused Approving
Result : The email that was actually sent was:

- **Recipient:** bob@example.com  
- **Subject:** “Running late”  
- **Body:** “I’ll be 10 minutes late.”


In [91]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='fa790a2a-4f79-417e-9376-2f15babd51da'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with given parameters.', 'tool_calls': [{'id': 'fc_ca97d74f-7cb9-4358-9519-09b3ef899405', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 205, 'total_tokens': 263, 'completion_time': 0.124748011, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.008289032, 'prompt_tokens_details': None, 'queue_time': 0.053371497, 'total_time': 0.133037043}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a4be3b0989', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model